In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the dataset files
# We reference the exact test file name provided
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# 2. Define the answer options available
options = ['A', 'B', 'C', 'D', 'E']

submission_preds = []

# 3. Iterate through each row in the test dataset to generate a dummy rank
for idx, row in test_df.iterrows():
    # Calculate the string length of each option text
    # (Heuristic: Longer options often contain more detailed/correct information)
    lengths = []
    for opt in options:
        opt_text = str(row[opt])
        lengths.append(len(opt_text))
        
    # Sort the options based on length in descending order
    # np.argsort returns indices that would sort the array in ascending order
    # [::-1] reverses it to descending order
    sorted_indices = np.argsort(lengths)[::-1]
    
    # Map the top 3 sorted indices back to their corresponding letter labels ('A'-'E')
    top_3_letters = [options[i] for i in sorted_indices[:3]]
    
    # Format exactly as required by the competition: space-separated string
    prediction_string = " ".join(top_3_letters)
    submission_preds.append(prediction_string)

# 4. Create the final submission dataframe
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Prediction': submission_preds
})

# 5. Save out to CSV for Kaggle submission
submission_df.to_csv("submission.csv", index=False)

print("Baseline submission successfully created!")
print(submission_df.head())

In [ ]:
import pandas as pd

# 1. Load the training dataset
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# 2. Clean any whitespace and get the distribution counts
# .value_counts() counts occurrences of unique values in the column
answer_counts = train_df['answer'].str.strip().value_counts()

print("--- Answer Frequency Distribution ---")
print(answer_counts)
print("-------------------------------------")

# 3. Identify the most and least frequent option counts
most_frequent_count = answer_counts.max()
least_frequent_count = answer_counts.min()

most_frequent_letter = answer_counts.idxmax()
least_frequent_letter = answer_counts.idxmin()

# 4. Calculate the sum
sum_occurrences = most_frequent_count + least_frequent_count

print(f"Most frequent option: '{most_frequent_letter}' with {most_frequent_count} occurrences.")
print(f"Least frequent option: '{least_frequent_letter}' with {least_frequent_count} occurrences.")
print(f"Sum of most and least frequent options: {sum_occurrences}")

In [ ]:
import pandas as pd
import string

# 2. Define the cleaning and tokenization function
def get_cleaned_tokens(text):
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove all standard punctuation characters
    # string.punctuation contains: !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)
    
    # Split by whitespace
    tokens = text.split()
    return tokens

# 3. Aggregate all unique words into a Set
vocabulary = set()
for prompt in train_df['prompt']:
    tokens = get_cleaned_tokens(prompt)
    vocabulary.update(tokens)

# 4. Output the result
print(f"Total number of unique words (Vocabulary Size): {len(vocabulary)}")

In [ ]:
import pandas as pd
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# 2. Extract Row ID 1 prompt (0th index in Python)
prompt_1 = train_df.iloc[0]['prompt']

# 3. Clean and tokenize text
prompt_cleaned = prompt_1.lower().translate(str.maketrans('', '', string.punctuation))
words = prompt_cleaned.split()

# 4. Filter out standard sklearn stop words
remaining_words = [word for word in words if word not in ENGLISH_STOP_WORDS]

# 5. Output results
print(f"Remaining Words: {remaining_words}")
print(f"Total words left: {len(remaining_words)}")

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 2. Combine all textual columns into a single flat list
combined_texts = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    combined_texts.extend(train_df[col].astype(str).tolist())

# 3. Initialize and fit the TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)

# 4. Extract total feature columns (vocabulary size)
vocab_size = len(vectorizer.vocabulary_)
print(f"Exact total number of feature columns: {vocab_size}")

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 2. Re-fit the TfidfVectorizer on all combined prompt and option texts
combined_texts = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    combined_texts.extend(train_df[col].astype(str).tolist())

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)

# 3. Extract the prompt and option A for Row ID 1 (0th index in Python)
row_1 = train_df.iloc[0]
prompt_text = str(row_1['prompt'])
option_A_text = str(row_1['A'])

# 4. Transform the strings into TF-IDF vectors
prompt_vector = vectorizer.transform([prompt_text])
option_A_vector = vectorizer.transform([option_A_text])

# 5. Compute the Cosine Similarity
similarity_score = cosine_similarity(prompt_vector, option_A_vector)[0][0]

# 6. Format and display the output
print(f"Cosine Similarity Score: {similarity_score:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 2. Fit the TfidfVectorizer on all text fields combined
combined_texts = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    combined_texts.extend(train_df[col].astype(str).tolist())

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_texts)

# 3. Track matches
correct_matches = 0
options = ['A', 'B', 'C', 'D', 'E']

# 4. Iterate over every row
for idx, row in train_df.iterrows():
    # Transform the prompt
    prompt_vector = vectorizer.transform([str(row['prompt'])])
    
    # Calculate similarity scores for all 5 options
    scores = []
    for opt in options:
        opt_vector = vectorizer.transform([str(row[opt])])
        sim_score = cosine_similarity(prompt_vector, opt_vector)[0][0]
        scores.append(sim_score)
        
    # Determine which option has the highest similarity
    best_option_index = np.argmax(scores)
    predicted_letter = options[best_option_index]
    
    # Check if the highest similarity matches the true answer label
    if predicted_letter == str(row['answer']).strip():
        correct_matches += 1

# 5. Calculate and display the final percentage
match_percentage = (correct_matches / len(train_df)) * 100
print(f"Total Rows Evaluated: {len(train_df)}")
print(f"Number of Matches: {correct_matches}")
print(f"Percentage of instances matching correct answer: {match_percentage:.2f}%")